# Bronze to Silver ETL Demo Notebook

This notebook demonstrates how to transform Bronze layer data into a Silver layer Star Schema.

**Transformations:**
- `dim_products` - Product dimension table (denormalized: merging products + aisles + departments)
- `dim_users` - User dimension table (aggregated from orders)
- `fact_orders` - Orders fact table (with pre-computed metrics)
- `fact_order_products` - Order-product fact table (Factless Fact Table)

## 1. Configure Env

In [ ]:
%idle_timeout 60
%glue_version 4.0
%worker_type G.1X
%number_of_workers 2
%additional_python_modules 
%extra_jars 
%%configure
{
    "--datalake-formats": "iceberg",
    "--conf": "spark.sql.extensions=org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions --conf spark.sql.catalog.glue_catalog=org.apache.iceberg.spark.SparkCatalog --conf spark.sql.catalog.glue_catalog.catalog-impl=org.apache.iceberg.aws.glue.GlueCatalog --conf spark.sql.catalog.glue_catalog.io-impl=org.apache.iceberg.aws.s3.S3FileIO --conf spark.sql.catalog.glue_catalog.warehouse=s3://instacart-aws-data-ml-eng-project/iceberg-warehouse/"
}

In [ ]:
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session

print("Spark version:", spark.version)
print("Initialization complete!")

In [ ]:
BUCKET_NAME = "instacart-aws-data-ml-eng-project"
CATALOG_NAME = "glue_catalog"
BRONZE_DB = "bronze"
SILVER_DB = "silver"

## 2. Create Silver Database

In [ ]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {CATALOG_NAME}.{SILVER_DB} LOCATION 's3://{BUCKET_NAME}/{SILVER_DB}/'")
spark.sql(f"SHOW DATABASES IN {CATALOG_NAME}").show()

## 3. Read Bronze Tables

In [ ]:
# Read all bronze tables
orders = spark.table(f"{CATALOG_NAME}.{BRONZE_DB}.orders")
products = spark.table(f"{CATALOG_NAME}.{BRONZE_DB}.products")
aisles = spark.table(f"{CATALOG_NAME}.{BRONZE_DB}.aisles")
departments = spark.table(f"{CATALOG_NAME}.{BRONZE_DB}.departments")
order_products = spark.table(f"{CATALOG_NAME}.{BRONZE_DB}.order_products")

print(f"orders: {orders.count()} rows")
print(f"products: {products.count()} rows")
print(f"aisles: {aisles.count()} rows")
print(f"departments: {departments.count()} rows")
print(f"order_products: {order_products.count()} rows")

In [ ]:
# Preview bronze tables
orders.show(5)
orders.printSchema()

## 4. Build Dimension Tables

### 4.1 dim_products (Product Dimension - Denormalized)

Merge products + aisles + departments into one table to avoid multi-table JOINs at query time

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG_NAME}.{SILVER_DB}.dim_products
USING iceberg
TBLPROPERTIES ('format-version' = '2')
AS
SELECT
    p.product_id,
    p.product_name,
    p.aisle_id,
    a.aisle AS aisle_name,
    p.department_id,
    d.department AS department_name,
    current_timestamp() AS audit_transform_time
FROM {CATALOG_NAME}.{BRONZE_DB}.products p
LEFT JOIN {CATALOG_NAME}.{BRONZE_DB}.aisles a ON p.aisle_id = a.aisle_id
LEFT JOIN {CATALOG_NAME}.{BRONZE_DB}.departments d ON p.department_id = d.department_id
""")
print(f"✅ Created {CATALOG_NAME}.{SILVER_DB}.dim_products")

### 4.2 dim_users (User Dimension - Aggregated)

Aggregate user features from orders table to avoid real-time computation at query time

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG_NAME}.{SILVER_DB}.dim_users
USING iceberg
TBLPROPERTIES ('format-version' = '2')
AS
WITH first_order AS (
    SELECT
        user_id,
        order_dow AS first_order_dow,
        order_hour_of_day AS first_order_hour
    FROM {CATALOG_NAME}.{BRONZE_DB}.orders
    WHERE order_number = 1
),
last_order AS (
    SELECT
        o.user_id,
        o.order_dow AS last_order_dow,
        o.order_hour_of_day AS last_order_hour
    FROM {CATALOG_NAME}.{BRONZE_DB}.orders o
    INNER JOIN (
        SELECT user_id, MAX(order_number) AS max_order_number
        FROM {CATALOG_NAME}.{BRONZE_DB}.orders
        GROUP BY user_id
    ) m ON o.user_id = m.user_id AND o.order_number = m.max_order_number
)
SELECT
    o.user_id,
    COUNT(o.order_id)             AS total_orders,
    MAX(o.order_number)           AS max_order_number,
    AVG(o.days_since_prior_order) AS avg_days_between_orders,
    MIN(o.days_since_prior_order) AS min_days_between_orders,
    MAX(o.days_since_prior_order) AS max_days_between_orders,
    f.first_order_dow,
    f.first_order_hour,
    l.last_order_dow,
    l.last_order_hour,
    current_timestamp()           AS audit_transform_time
FROM {CATALOG_NAME}.{BRONZE_DB}.orders o
LEFT JOIN first_order f ON o.user_id = f.user_id
LEFT JOIN last_order l ON o.user_id = l.user_id
GROUP BY o.user_id, f.first_order_dow, f.first_order_hour, l.last_order_dow, l.last_order_hour
""")
print(f"✅ Created {CATALOG_NAME}.{SILVER_DB}.dim_users")

## 5. Build Fact Tables

### 5.1 fact_orders (Orders Fact Table - With Pre-computed Metrics)

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG_NAME}.{SILVER_DB}.fact_orders
USING iceberg
TBLPROPERTIES ('format-version' = '2')
AS
WITH order_metrics AS (
    SELECT
        order_id,
        COUNT(product_id) AS total_products,
        SUM(reordered)    AS total_reordered
    FROM {CATALOG_NAME}.{BRONZE_DB}.order_products
    GROUP BY order_id
)
SELECT
    o.order_id,
    o.user_id,
    o.eval_set,
    o.order_number,
    o.order_dow,
    o.order_hour_of_day,
    o.days_since_prior_order,
    m.total_products,
    m.total_reordered,
    CASE
        WHEN m.total_products > 0 THEN m.total_reordered / m.total_products
        ELSE 0
    END AS reorder_ratio,
    current_timestamp() AS audit_transform_time
FROM {CATALOG_NAME}.{BRONZE_DB}.orders o
LEFT JOIN order_metrics m ON o.order_id = m.order_id
""")
print(f"✅ Created {CATALOG_NAME}.{SILVER_DB}.fact_orders")

### 5.2 fact_order_products (Order-Product Fact Table - Factless Fact)

This is a Factless Fact Table with no traditional measures (amount/quantity), but records the "purchase" business event

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {CATALOG_NAME}.{SILVER_DB}.fact_order_products
USING iceberg
TBLPROPERTIES ('format-version' = '2')
AS
SELECT
    op.order_id,
    op.product_id,
    o.user_id,
    op.add_to_cart_order,
    op.reordered,
    current_timestamp() AS audit_transform_time
FROM {CATALOG_NAME}.{BRONZE_DB}.order_products op
LEFT JOIN {CATALOG_NAME}.{BRONZE_DB}.orders o ON op.order_id = o.order_id
""")
print(f"✅ Created {CATALOG_NAME}.{SILVER_DB}.fact_order_products")

## 7. Verify Silver Tables

In [ ]:
# List all tables in silver database
spark.sql(f"SHOW TABLES IN {CATALOG_NAME}.{SILVER_DB}").show()

In [ ]:
# Verify dim_products
spark.table(f"{CATALOG_NAME}.{SILVER_DB}.dim_products").show(5)
spark.sql(f"SELECT COUNT(*) as total FROM {CATALOG_NAME}.{SILVER_DB}.dim_products").show()

In [ ]:
# Verify fact_orders
spark.table(f"{CATALOG_NAME}.{SILVER_DB}.fact_orders").show(5)
spark.sql(f"SELECT COUNT(*) as total FROM {CATALOG_NAME}.{SILVER_DB}.fact_orders").show()

## 8. Sample Analytics Queries

Validate the Silver layer modeling: simple SQL can answer business questions

In [ ]:
# Query 1: 各部门的产品销量排名
spark.sql(f"""
    SELECT 
        p.department_name,
        COUNT(*) as total_sold
    FROM {CATALOG_NAME}.{SILVER_DB}.fact_order_products f
    JOIN {CATALOG_NAME}.{SILVER_DB}.dim_products p ON f.product_id = p.product_id
    GROUP BY p.department_name
    ORDER BY total_sold DESC
""").show()

In [ ]:
# Query 2: Weekend vs Weekday order distribution
spark.sql(f"""
    SELECT 
        (order_dow IN (0, 6)) AS is_weekend,
        COUNT(*) as total_orders,
        AVG(total_products) as avg_products_per_order
    FROM {CATALOG_NAME}.{SILVER_DB}.fact_orders
    GROUP BY 1
""").show()

In [ ]:
# Query 3: 复购率最高的 Top 10 产品
spark.sql(f"""
    SELECT 
        p.product_name,
        p.department_name,
        COUNT(*) as times_purchased,
        SUM(f.reordered) as times_reordered,
        ROUND(SUM(f.reordered) * 100.0 / COUNT(*), 2) as reorder_rate
    FROM {CATALOG_NAME}.{SILVER_DB}.fact_order_products f
    JOIN {CATALOG_NAME}.{SILVER_DB}.dim_products p ON f.product_id = p.product_id
    GROUP BY p.product_name, p.department_name
    HAVING COUNT(*) >= 100
    ORDER BY reorder_rate DESC
    LIMIT 10
""").show(truncate=False)

In [ ]:
spark.sql(f"""
    SELECT 
        CASE 
            WHEN total_orders = 1 THEN '1_One-time Buyer'
            WHEN total_orders BETWEEN 2 AND 5 THEN '2_Low Frequency (2-5 orders)'
            WHEN total_orders BETWEEN 6 AND 15 THEN '3_Medium Frequency (6-15 orders)'
            ELSE '4_High Frequency (>15 orders)'
        END as user_segment,
        COUNT(*) as user_count,
        ROUND(AVG(avg_days_between_orders), 1) as avg_purchase_interval
    FROM {CATALOG_NAME}.{SILVER_DB}.dim_users
    GROUP BY 1
    ORDER BY 1
""").show()

## 9. Clean Up (Optional)

In [ ]:
# Uncomment to drop all silver tables
# spark.sql(f"DROP TABLE IF EXISTS {CATALOG_NAME}.{SILVER_DB}.dim_products")
# spark.sql(f"DROP TABLE IF EXISTS {CATALOG_NAME}.{SILVER_DB}.dim_users")
# spark.sql(f"DROP TABLE IF EXISTS {CATALOG_NAME}.{SILVER_DB}.fact_orders")
# spark.sql(f"DROP TABLE IF EXISTS {CATALOG_NAME}.{SILVER_DB}.fact_order_products")
# spark.sql(f"DROP DATABASE IF EXISTS {CATALOG_NAME}.{SILVER_DB}")